# AWS Cost & Usage Assistant

Uses OpenAI function calling to answer natural-language questions about AWS spend by invoking Cost Explorer and EC2 APIs through boto3.

In [30]:
import os
import json
import datetime
import boto3
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

MODEL = "gpt-4o"
openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


## Verify AWS credentials

Confirm boto3 is picking up credentials and check which IAM identity is active.

In [31]:
session = boto3.Session()
creds = session.get_credentials()
print("Credential source:", creds.method)

Credential source: shared-credentials-file


## Tool functions

Each function wraps a boto3 call that the model can invoke via function calling.

In [32]:
def get_aws_cost(days: int = 7) -> str:
    ce = boto3.client("ce")
    end = datetime.date.today()
    start = end - datetime.timedelta(days=days)
    resp = ce.get_cost_and_usage(
        TimePeriod={"Start": start.isoformat(), "End": end.isoformat()},
        Granularity="DAILY",
        Metrics=["UnblendedCost"],
    )
    return json.dumps(resp["ResultsByTime"])


def get_cost_by_service(days: int = 7) -> str:
    ce = boto3.client("ce")
    end = datetime.date.today()
    start = end - datetime.timedelta(days=days)
    resp = ce.get_cost_and_usage(
        TimePeriod={"Start": start.isoformat(), "End": end.isoformat()},
        Granularity="DAILY",
        Metrics=["UnblendedCost"],
        GroupBy=[{"Type": "DIMENSION", "Key": "SERVICE"}],
    )
    return json.dumps(resp["ResultsByTime"])


def get_s3_storage_cost(days: int = 7) -> str:
    ce = boto3.client("ce")
    end = datetime.date.today()
    start = end - datetime.timedelta(days=days)
    resp = ce.get_cost_and_usage(
        TimePeriod={"Start": start.isoformat(), "End": end.isoformat()},
        Granularity="DAILY",
        Metrics=["UnblendedCost"],
        Filter={"Dimensions": {"Key": "SERVICE", "Values": ["Amazon Simple Storage Service"]}},
        GroupBy=[{"Type": "DIMENSION", "Key": "USAGE_TYPE"}],
    )
    return json.dumps(resp["ResultsByTime"])


def list_ec2_instances() -> str:
    ec2 = boto3.client("ec2")
    resp = ec2.describe_instances()
    instances = [
        {"id": i["InstanceId"], "type": i["InstanceType"], "state": i["State"]["Name"]}
        for r in resp["Reservations"] for i in r["Instances"]
    ]
    return json.dumps(instances)


TOOL_FUNCTIONS = {
    "get_aws_cost": get_aws_cost,
    "get_cost_by_service": get_cost_by_service,
    "get_s3_storage_cost": get_s3_storage_cost,
    "list_ec2_instances": list_ec2_instances,
}

## Tool schema

JSON schema definitions the model uses to decide which function to call and with what arguments.

In [37]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_aws_cost",
            "description": "Get total AWS unblended cost for the last N days from Cost Explorer",
            "parameters": {
                "type": "object",
                "properties": {"days": {"type": "integer", "description": "days to look back"}},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_cost_by_service",
            "description": "Get AWS unblended cost for the last N days, broken down by service",
            "parameters": {
                "type": "object",
                "properties": {"days": {"type": "integer", "description": "days to look back"}},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_s3_storage_cost",
            "description": "Get S3 storage cost breakdown for the last N days",
            "parameters": {
                "type": "object",
                "properties": {"days": {"type": "integer", "description": "days to look back"}},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "list_ec2_instances",
            "description": "List all EC2 instances in the account with their type and current state",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
]

## Single tool-call demo

Ask a cost question; the model picks `get_aws_cost` and we execute it once.

In [35]:
messages = [{"role": "user", "content": "Why did my AWS spend spike this week?"}]

resp = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
msg = resp.choices[0].message

if msg.tool_calls:
    messages.append(msg)
    for call in msg.tool_calls:
        args = json.loads(call.function.arguments)
        result = TOOL_FUNCTIONS[call.function.name](**args)
        messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

    final = openai.chat.completions.create(model=MODEL, messages=messages)
    print(final.choices[0].message.content)
else:
    print(msg.content)

Based on the recent cost data from your AWS account, here's what I found:

1. **Overall Cost**: There was a significant spike in your AWS spending on July 14 and July 15, 2026, with costs of $0.020 and $0.050 USD, respectively, compared to marginal usage on other days.

2. **Service Breakdown**:
   - **AWS Cost Explorer**: This service contributed to the majority of the spike. Costs of $0.02 USD on July 14 and $0.05 USD on July 15.
   - **Amazon Simple Storage Service (S3)**: The cost remained minimal throughout the period, indicating S3 wasn't a major contributor.
   - **Other Services**: AWS Glue and Amazon CloudWatch reported no significant costs.

3. **S3 Storage Cost**:
   - There were negligible costs related to S3 storage and request operations, around $0.000008115 USD consistently.
   - Minimal increases related to request operations were noted on July 15 and 16, with tiny costs corresponding to Tier 1 requests.

4. **EC2 Instances**: There are no active EC2 instances in the ac

## Agentic loop

General loop that keeps calling tools until the model responds with plain text.

In [38]:
messages = [{"role": "user", "content": "Is my AWS spend normal, and what's actually running right now?"}]

TOOL_FUNCTIONS = {
    "get_aws_cost": get_aws_cost,
    "get_cost_by_service": get_cost_by_service,
    "get_s3_storage_cost": get_s3_storage_cost,
    "list_ec2_instances": list_ec2_instances,
} 

while True:
    resp = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    msg = resp.choices[0].message
    messages.append(msg)

    if not msg.tool_calls:
        print(msg.content)
        break

    for call in msg.tool_calls:
        args = json.loads(call.function.arguments)
        result = TOOL_FUNCTIONS[call.function.name](**args)
        messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

Here's an overview of your AWS activity:

### AWS Spend Overview
Your AWS costs for the last 30 days include:

- **Total Costs and Daily Breakdown:**
  - June 20-30, 2026: Costs fluctuated between $0.000008 and $0.000018 USD daily.
  - July 1-10, 2026: Similar minor fluctuations with daily estimates ranging similarly.
  - **July 14-16, 2026**: Your costs peaked significantly at $0.020 and $0.050 USD for a couple of days.

### Current AWS Instances:
- No EC2 instances are listed as running at this time.

If you need any analysis on specific services contributing to these costs or if there are any other AWS resources to monitor, feel free to ask!


In [ ]:
def ask_agent(question: str, verbose: bool = True) -> str:
    messages = [{"role": "user", "content": question}]

    while True:
        resp = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        msg = resp.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            return msg.content

        for call in msg.tool_calls:
            name = call.function.name
            args = json.loads(call.function.arguments)
            if verbose:
                print(f"  → calling {name}({args})")

            if name not in TOOL_FUNCTIONS:
                result = f"Error: model requested unknown tool '{name}'"
            else:
                try:
                    result = TOOL_FUNCTIONS[name](**args)
                except Exception as e:
                    result = f"Error running '{name}': {e}"
            print(f"     result: {result}")   # <-- add this line

            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

In [40]:
test_questions = [
    "What's driving my AWS spend this week?",
    "Is anything running that I forgot about?",
    "What did I spend last month?",
    "Do I have any S3 costs?",
    "Are my EC2 costs matching the number of instances actually running?",
]

for q in test_questions:
    print(f"\nQ: {q}")
    answer = ask_agent(q)
    print(f"A: {answer}")


Q: What's driving my AWS spend this week?
  → calling get_aws_cost({'days': 7})
  → calling get_cost_by_service({'days': 7})
A: Here's a breakdown of your AWS spend for the past week:

### Total Costs:
- **2026-07-13 to 2026-07-14**: $0.000008115
- **2026-07-14 to 2026-07-15**: $0.020008115
- **2026-07-15 to 2026-07-16**: $0.050013115
- **2026-07-16 to 2026-07-17**: $0.0500239192
- **2026-07-17 to 2026-07-18**: $0.000008115
- **2026-07-18 to 2026-07-19**: $0.000008115
- **2026-07-19 to 2026-07-20**: $0.000008115

### Cost by Service:

#### 2026-07-14 to 2026-07-15:
- **AWS Cost Explorer**: $0.020
- **Amazon Simple Storage Service (S3)**: $0.000008115

#### 2026-07-15 to 2026-07-16:
- **AWS Cost Explorer**: $0.050
- **Amazon Simple Storage Service (S3)**: $0.000013115

#### 2026-07-16 to 2026-07-17:
- **AWS Cost Explorer**: $0.050
- **Amazon Simple Storage Service (S3)**: $0.0000239192

Overall, your main costs this week are driven by usage of **AWS Cost Explorer** and **Amazon Simple 